# Gold Layer Tables

In [2]:
# ============================================================
# dim_vehicle — column-list typo fixed, dead merge cell removed,
# customer_sk attached positionally (valid now that Silver is
# aligned), premium_vehicle redefined since make is a numeric
# code, not a brand name
# ============================================================

import pandas as pd
from pathlib import Path

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold/dimensions")
GOLD_PATH.mkdir(parents=True, exist_ok=True)

vehicle = pd.read_csv(SILVER_PATH / "vehicle.csv")
safety = pd.read_csv(SILVER_PATH / "vehicle_safety.csv")
customer = pd.read_csv(SILVER_PATH / "customer.csv")

dim_vehicle = vehicle.merge(safety, on="policy_id", how="left")
assert len(dim_vehicle) == len(customer), "vehicle/customer row counts diverged — alignment broke somewhere upstream"

# positional attach — valid because Silver's customer/vehicle/policy/claim
# are the same aligned population in the same row order
dim_vehicle["customer_sk"] = customer["customer_sk"].values

dim_vehicle["vehicle_age_band"] = dim_vehicle["vehicle_age"].apply(
    lambda a: "New" if a <= 2 else "Mid Age" if a <= 5 else "Old"
)
dim_vehicle["weight_class"] = dim_vehicle["gross_weight"].apply(
    lambda w: "Light" if w < 1000 else "Medium" if w <= 1500 else "Heavy"
)
dim_vehicle["engine_category"] = dim_vehicle["displacement"].apply(
    lambda cc: "Small" if cc < 1000 else "Medium" if cc <= 1500 else "Large"
)
dim_vehicle["safety_rating"] = dim_vehicle["safety_score"].apply(
    lambda s: "Excellent" if s >= 90 else "Good" if s >= 75 else "Average" if s >= 60 else "Poor"
)
vehicle_area = dim_vehicle["length"] * dim_vehicle["width"]
dim_vehicle["vehicle_size"] = vehicle_area.apply(
    lambda a: "Compact" if a < 5000000 else "Mid Size" if a < 6500000 else "Large"
)

# make is a numeric code in this dataset, not a brand — a brand-name
# check can never match. Redefine premium_vehicle on real signals instead.
# Thresholds are a starting point — tune once you've seen the actual
# displacement/ncap distribution.
dim_vehicle["premium_vehicle"] = (
    (dim_vehicle["displacement"] >= 1200) & (dim_vehicle["ncap_rating"] >= 3)
)

dim_vehicle["transmission_category"] = dim_vehicle["transmission"].replace(
    {"Automatic": "Automatic", "Manual": "Manual"}
)

dim_vehicle = dim_vehicle[
    [
        "vehicle_sk", "policy_id", "customer_sk",
        "make", "model", "segment",
        "fuel_type",
        "vehicle_age", "vehicle_age_band",
        "engine_type", "displacement", "engine_category",
        "cylinder",
        "transmission", "transmission_category",
        "steering",
        "length", "width", "height",
        "gross_weight", "weight_class",
        "vehicle_size",
        "airbags", "is_esc", "is_tpms", "is_parking_sensors",
        "is_parking_camera", "is_brake_assist", "is_power_steering", "is_speed_alert",
        "ncap_rating", "safety_score", "safety_rating",
        "premium_vehicle"
    ]
]

dim_vehicle.to_csv(GOLD_PATH / "dim_vehicle.csv", index=False)
dim_vehicle.to_parquet(GOLD_PATH / "dim_vehicle.parquet", index=False)
print("dim_vehicle:", dim_vehicle.shape)


# ============================================================
# dim_policy — renewal_probability/policy_age_score removed
# (they're model outputs, now live in fact_underwriting only),
# tenure_category duplicate of policy_tenure_band dropped
# ============================================================

policy = pd.read_csv(SILVER_PATH / "policy.csv")

policy["active_flag"] = policy["policy_status"].str.strip().str.lower().eq("active")

policy = policy[
    [
        "policy_sk", "policy_id",
        "policy_type", "coverage_type",
        "policy_status", "active_flag",
        "policy_tenure", "policy_tenure_band",
        "source_system", "load_date"
    ]
]

policy.to_csv(GOLD_PATH / "dim_policy.csv", index=False)
policy.to_parquet(GOLD_PATH / "dim_policy.parquet", index=False)
print("dim_policy:", policy.shape)


# ============================================================
# fact_quote — dead merges removed (vehicle_sk already comes
# straight from Silver; policy_sk attached positionally), 
# quote_value_band rescaled to match the actual premium formula
# ============================================================

FACT_PATH = Path("../data/gold/facts")
FACT_PATH.mkdir(parents=True, exist_ok=True)

quote = pd.read_csv(SILVER_PATH / "quote.csv")
channel = pd.read_csv(GOLD_PATH / "dim_channel.csv")

quote["sales_channel"] = pd.to_numeric(quote["sales_channel"], errors="coerce").astype("Int64")
channel["channel_id"] = channel["channel_id"].astype(int)
quote["quote_date"] = pd.to_datetime(quote["quote_date"])
quote["date_sk"] = quote["quote_date"].dt.strftime("%Y%m%d").astype(int)

quote = quote.merge(
    channel[["channel_sk", "channel_id"]],
    left_on="sales_channel", right_on="channel_id", how="left"
)

# vehicle_sk already exists on quote.csv from Silver — no merge needed.
# policy_sk attached positionally, same aligned population.
quote["policy_sk"] = policy["policy_sk"].values

# rescaled to the real premium range (~3,500-8,000) instead of bins
# that assumed premiums up to 20,000+ — the old bins left "High" and
# "Premium" essentially empty
quote["quote_value_band"] = pd.cut(
    quote["quoted_premium"],
    bins=[0, 4500, 5500, 7000, float("inf")],
    labels=["Low", "Medium", "High", "Premium"]
)

fact_quote = quote[
    [
        "quote_sk", "quote_id", "customer_sk", "vehicle_sk", "policy_sk",
        "channel_sk", "date_sk", "quoted_premium", "accepted_offer",
        "conversion_flag", "quote_status", "quote_stage",
        "device_type", "quote_source", "days_since_last_contact", "quote_value_band"
    ]
]

fact_quote.to_csv(FACT_PATH / "fact_quote.csv", index=False)
fact_quote.to_parquet(FACT_PATH / "fact_quote.parquet", index=False)
print("fact_quote:", fact_quote.shape)


# ============================================================
# fact_claim — dead vehicle merge removed (claim.csv already
# has vehicle_sk/customer_sk from Silver), claim_date anchored
# to the customer's quote date instead of an independent random
# range, claim_approved given its own outcome instead of
# mirroring claim_flag
# ============================================================

import numpy as np
np.random.seed(42)

claim = pd.read_csv(SILVER_PATH / "claim.csv")
dim_date = pd.read_csv(GOLD_PATH / "dim_date.csv")
dim_date["date"] = pd.to_datetime(dim_date["date"])
max_date = dim_date["date"].max()

claim = claim.merge(
    policy[["policy_sk", "policy_id"]], on="policy_id", how="left"
)
# vehicle_sk/customer_sk already on claim.csv — no merge needed

claim = claim.merge(
    fact_quote[["customer_sk", "quote_date" if "quote_date" in fact_quote.columns else "date_sk"]],
    on="customer_sk", how="left"
)
# fact_quote doesn't carry quote_date post-select — rebuild from date_sk
claim["quote_date"] = pd.to_datetime(claim["date_sk"].astype(str), format="%Y%m%d")

# claims happen well into the policy period, not at quote time —
# lag of 30-700 days, capped to the warehouse's date range
lag_days = np.random.randint(30, 700, len(claim))
claim["claim_date"] = claim["quote_date"] + pd.to_timedelta(lag_days, unit="D")
claim["claim_date"] = claim["claim_date"].clip(upper=max_date)
claim["date_sk"] = claim["claim_date"].dt.strftime("%Y%m%d").astype(int)

# claim_approved now an independent outcome — filing a claim doesn't
# guarantee approval. ~85% approval rate for filed claims, tune once
# you see real settlement patterns.
claim["claim_approved"] = np.where(
    claim["claim_flag"] == 1,
    np.random.choice([True, False], len(claim), p=[0.85, 0.15]),
    False
)

claim["settlement_band"] = claim["settlement_days"].apply(
    lambda d: "Fast" if d <= 3 else "Standard" if d <= 7 else "Delayed"
)
claim["claim_amount_band"] = pd.cut(
    claim["claim_amount"], bins=[0, 25000, 50000, 100000, float("inf")],
    labels=["Low", "Medium", "High", "Very High"]
)

fact_claim = claim[
    [
        "claim_sk", "claim_id", "customer_sk", "vehicle_sk", "policy_sk",
        "date_sk", "claim_amount", "claim_amount_band", "claim_flag",
        "claim_approved", "claim_severity", "fraud_risk",
        "settlement_days", "settlement_band"
    ]
]

fact_claim.to_csv(FACT_PATH / "fact_claim.csv", index=False)
fact_claim.to_parquet(FACT_PATH / "fact_claim.parquet", index=False)
print("fact_claim:", fact_claim.shape)


# ============================================================
# fact_underwriting — date anchored to quote date + short TAT lag
# instead of independent random; recommendation (AI's raw
# suggestion) and underwriting_decision (final outcome) now
# genuinely diverge for referred cases, so an override rate is
# actually computable
# ============================================================

vehicle_dim = pd.read_csv(GOLD_PATH / "dim_vehicle.csv")
rows = len(policy)

underwriting = pd.DataFrame()
underwriting["underwriting_sk"] = range(1, rows + 1)
underwriting["policy_sk"] = policy["policy_sk"].values
underwriting["customer_sk"] = customer["customer_sk"].values
underwriting["vehicle_sk"] = vehicle_dim["vehicle_sk"].values

uw_quote_date = pd.to_datetime(fact_quote["date_sk"].astype(str), format="%Y%m%d")
tat_days = np.random.randint(0, 5, rows)  # underwriting turnaround: 0-4 days post-quote
uw_date = (uw_quote_date.values + pd.to_timedelta(tat_days, unit="D"))
uw_date = pd.Series(uw_date).clip(upper=max_date)
underwriting["underwriting_date_sk"] = uw_date.dt.strftime("%Y%m%d").astype(int)

underwriting["risk_score"] = np.random.randint(5, 100, rows)
underwriting["risk_band"] = underwriting["risk_score"].apply(
    lambda s: "Low" if s < 30 else "Medium" if s < 60 else "High" if s < 80 else "Very High"
)
underwriting["ai_confidence"] = np.round(np.random.uniform(75, 99.9, rows), 2)
underwriting["fraud_probability"] = np.round(underwriting["risk_score"] / 100, 2)

underwriting["recommendation"] = underwriting["risk_score"].apply(
    lambda s: "Auto Approve" if s < 35 else "Refer Underwriter" if s < 70 else "Reject"
)
underwriting["manual_review_flag"] = underwriting["recommendation"].eq("Refer Underwriter")

def final_decision(row):
    if row["recommendation"] == "Auto Approve":
        return "Approved"
    if row["recommendation"] == "Reject":
        return "Rejected"
    # referred cases: a human decides, weighted by where in the band the score sits
    p_approve = 0.60 if row["risk_score"] < 55 else 0.35
    return np.random.choice(["Approved", "Rejected"], p=[p_approve, 1 - p_approve])

underwriting["underwriting_decision"] = underwriting.apply(final_decision, axis=1)

underwriting["review_time_minutes"] = underwriting["manual_review_flag"].apply(
    lambda m: np.random.randint(20, 90) if m else np.random.randint(2, 8)
)
underwriting["premium_adjustment_pct"] = underwriting["risk_score"].apply(
    lambda s: np.random.randint(-10, 5) if s < 35 else np.random.randint(0, 15) if s < 70 else np.random.randint(15, 35)
)
underwriting["rules_triggered"] = np.random.randint(1, 8, rows)
underwriting["model_version"] = np.random.choice(
    ["UW_AI_v1.0", "UW_AI_v1.1", "UW_AI_v2.0"], rows, p=[0.25, 0.35, 0.40]
)
underwriting["underwriter"] = np.where(
    underwriting["manual_review_flag"],
    np.random.choice(["Alice", "John", "David", "Priya", "Rahul"], rows),
    "AI Engine"
)
underwriting["processing_mode"] = np.where(
    underwriting["manual_review_flag"], "Manual", "Straight Through Processing"
)
underwriting["sla_status"] = np.where(
    underwriting["review_time_minutes"] <= 30, "Within SLA", "SLA Breached"
)
underwriting["explanation_generated"] = True

underwriting = underwriting[
    [
        "underwriting_sk", "policy_sk", "customer_sk", "vehicle_sk",
        "underwriting_date_sk", "risk_score", "risk_band", "fraud_probability",
        "underwriting_decision", "recommendation", "manual_review_flag",
        "review_time_minutes", "premium_adjustment_pct", "rules_triggered",
        "ai_confidence", "processing_mode", "underwriter", "sla_status",
        "explanation_generated", "model_version"
    ]
]

underwriting.to_csv(FACT_PATH / "fact_underwriting.csv", index=False)
underwriting.to_parquet(FACT_PATH / "fact_underwriting.parquet", index=False)
print("fact_underwriting:", underwriting.shape)


# ============================================================
# fact_customer_journey — now keyed off conversion_flag instead
# of accepted_offer (matches the two-layer funnel we just built
# into fact_quote), date_sk anchored near the quote date
# ============================================================

import uuid

journey_flow = [
    ("Website Visit", "Visited Website"), ("Quote Started", "Started Quote"),
    ("Vehicle Details", "Entered Vehicle Details"), ("KYC Upload", "Uploaded Documents"),
    ("Premium Review", "Viewed Premium"), ("Payment", "Payment Attempt"),
    ("Policy Issued", "Policy Generated")
]
drop_reasons = ["Premium Too High", "Documents Missing", "Technical Error",
                 "Changed Mind", "Slow Response", "Switched Insurer"]
devices = ["Desktop", "Mobile", "Tablet"]
sources = ["Google", "Facebook", "Direct", "Email", "Partner", "Organic Search"]

records = []
journey_sk = 1

for _, row in fact_quote.iterrows():
    quote_date = pd.to_datetime(str(row["date_sk"]), format="%Y%m%d")
    device = np.random.choice(devices, p=[0.30, 0.60, 0.10])
    source = np.random.choice(sources)
    ai_used = np.random.choice([True, False], p=[0.45, 0.55])

    # conversion_flag, not accepted_offer — a customer can accept the
    # quote and still not end up with an issued policy
    converted = bool(row["conversion_flag"])
    completed_stage = len(journey_flow) if converted else np.random.randint(2, 7)
    session = str(uuid.uuid4())[:12]

    for stage_no, (stage, action) in enumerate(journey_flow, start=1):
        completed = stage_no <= completed_stage
        abandoned = (not converted) and (stage_no == completed_stage)
        duration = np.random.randint(15, 240)
        exit_reason = np.random.choice(drop_reasons) if abandoned else None

        event_date = quote_date + pd.Timedelta(minutes=int(stage_no * np.random.randint(5, 60)))
        event_date = min(event_date, max_date)

        records.append({
            "journey_sk": journey_sk, "customer_sk": row["customer_sk"],
            "quote_sk": row["quote_sk"], "date_sk": int(event_date.strftime("%Y%m%d")),
            "session_id": session, "stage_sequence": stage_no, "stage_name": stage,
            "customer_action": action, "duration_seconds": duration,
            "completed_flag": completed, "abandoned_flag": abandoned,
            "exit_reason": exit_reason, "device_type": device, "traffic_source": source,
            "ai_assistance_used": ai_used, "conversion_flag": converted
        })
        journey_sk += 1
        if abandoned:
            break

fact_customer_journey = pd.DataFrame(records)
fact_customer_journey["duration_band"] = pd.cut(
    fact_customer_journey["duration_seconds"], bins=[0, 30, 60, 120, 300],
    labels=["Very Fast", "Fast", "Average", "Slow"]
)
fact_customer_journey["journey_status"] = np.where(
    fact_customer_journey["abandoned_flag"], "Abandoned",
    np.where(fact_customer_journey["conversion_flag"], "Converted", "In Progress")
)

fact_customer_journey.to_csv(FACT_PATH / "fact_customer_journey.csv", index=False)
fact_customer_journey.to_parquet(FACT_PATH / "fact_customer_journey.parquet", index=False)
print("fact_customer_journey:", fact_customer_journey.shape)

dim_vehicle: (97655, 34)
dim_policy: (97655, 10)
fact_quote: (97655, 16)
fact_claim: (97655, 14)
fact_underwriting: (97655, 20)
fact_customer_journey: (419906, 18)


# Dim_date

In [2]:
import pandas as pd
from pathlib import Path

# ==========================================================
# Paths
# ==========================================================

GOLD = Path("../data/gold")
DIM = GOLD / "dimensions"

DIM.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Date Range
# ==========================================================

start_date = "2022-01-01"
end_date = "2026-12-31"

date = pd.DataFrame(
    {
        "date": pd.date_range(
            start=start_date,
            end=end_date,
            freq="D"
        )
    }
)

# ==========================================================
# Surrogate Key
# ==========================================================

date["date_sk"] = (
    date["date"]
    .dt.strftime("%Y%m%d")
    .astype(int)
)

# ==========================================================
# Calendar Attributes
# ==========================================================

date["year"] = date["date"].dt.year

date["quarter"] = date["date"].dt.quarter

date["quarter_name"] = (
    "Q" + date["quarter"].astype(str)
)

date["month"] = date["date"].dt.month

date["month_name"] = date["date"].dt.month_name()

date["month_short"] = date["date"].dt.strftime("%b")

date["week"] = date["date"].dt.isocalendar().week.astype(int)

date["week_of_month"] = (
    ((date["date"].dt.day - 1) // 7) + 1
)

date["day"] = date["date"].dt.day

date["day_name"] = date["date"].dt.day_name()

date["day_of_week"] = date["date"].dt.weekday + 1

date["day_of_year"] = date["date"].dt.dayofyear

# ==========================================================
# Weekend Flags
# ==========================================================

date["is_weekend"] = (
    date["day_name"]
    .isin(
        [
            "Saturday",
            "Sunday"
        ]
    )
)

# ==========================================================
# Month Flags
# ==========================================================

date["is_month_start"] = date["date"].dt.is_month_start

date["is_month_end"] = date["date"].dt.is_month_end

# ==========================================================
# Quarter Flags
# ==========================================================

date["is_quarter_start"] = date["date"].dt.is_quarter_start

date["is_quarter_end"] = date["date"].dt.is_quarter_end

# ==========================================================
# Year Flags
# ==========================================================

date["is_year_start"] = date["date"].dt.is_year_start

date["is_year_end"] = date["date"].dt.is_year_end

# ==========================================================
# Financial Year (India)
# ==========================================================

date["financial_year"] = date["year"]

date.loc[
    date["month"] <= 3,
    "financial_year"
] = date["year"] - 1

date["financial_year"] = (
    date["financial_year"].astype(str)
    + "-"
    + (date["financial_year"] + 1).astype(str)
)

# ==========================================================
# Financial Quarter (India)
# ==========================================================

def financial_quarter(month):

    if month in [4,5,6]:
        return "Q1"

    elif month in [7,8,9]:
        return "Q2"

    elif month in [10,11,12]:
        return "Q3"

    else:
        return "Q4"

date["financial_quarter"] = date["month"].apply(financial_quarter)

# ==========================================================
# Reorder Columns
# ==========================================================

date = date[
    [

        "date_sk",

        "date",

        "year",

        "quarter",

        "quarter_name",

        "month",

        "month_name",

        "month_short",

        "week",

        "week_of_month",

        "day",

        "day_name",

        "day_of_week",

        "day_of_year",

        "is_weekend",

        "is_month_start",

        "is_month_end",

        "is_quarter_start",

        "is_quarter_end",

        "is_year_start",

        "is_year_end",

        "financial_year",

        "financial_quarter"

    ]
]

# ==========================================================
# Save
# ==========================================================

date.to_csv(
    DIM / "dim_date.csv",
    index=False
)

date.to_parquet(
    DIM / "dim_date.parquet",
    index=False
)

print("=" * 60)
print("DIM_DATE CREATED SUCCESSFULLY")
print("=" * 60)

print(date.head())

print(f"\nRows    : {len(date):,}")
print(f"Columns : {len(date.columns)}")

DIM_DATE CREATED SUCCESSFULLY
    date_sk       date  year  quarter quarter_name  month month_name  \
0  20220101 2022-01-01  2022        1           Q1      1    January   
1  20220102 2022-01-02  2022        1           Q1      1    January   
2  20220103 2022-01-03  2022        1           Q1      1    January   
3  20220104 2022-01-04  2022        1           Q1      1    January   
4  20220105 2022-01-05  2022        1           Q1      1    January   

  month_short  week  week_of_month  ...  day_of_year is_weekend  \
0         Jan    52              1  ...            1       True   
1         Jan    52              1  ...            2       True   
2         Jan     1              1  ...            3      False   
3         Jan     1              1  ...            4      False   
4         Jan     1              1  ...            5      False   

   is_month_start  is_month_end  is_quarter_start  is_quarter_end  \
0            True         False              True           False

# dim_channel

In [1]:
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold/dimensions")

GOLD_PATH.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Load Silver Table
# ------------------------------------------------------------------

channel = pd.read_csv(SILVER_PATH / "channel_lookup.csv")

# ------------------------------------------------------------------
# Create Surrogate Key
# ------------------------------------------------------------------

channel.insert(0, "channel_sk", range(1, len(channel) + 1))

# ------------------------------------------------------------------
# Channel Type
# ------------------------------------------------------------------

online_channels = [
    "Website",
    "Mobile App",
    "Aggregator"
]

channel["channel_type"] = channel["channel_name"].apply(
    lambda x: "Online" if x in online_channels else "Offline"
)

# ------------------------------------------------------------------
# Ownership
# ------------------------------------------------------------------

direct_channels = [
    "Website",
    "Mobile App",
    "Branch",
    "Call Center"
]

channel["ownership"] = channel["channel_name"].apply(
    lambda x: "Direct" if x in direct_channels else "Indirect"
)

# ------------------------------------------------------------------
# Digital Flag
# ------------------------------------------------------------------

digital_channels = [
    "Website",
    "Mobile App"
]

channel["is_digital"] = channel["channel_name"].isin(digital_channels)

# ------------------------------------------------------------------
# Priority
# ------------------------------------------------------------------

priority = {
    "Website": "High",
    "Mobile App": "High",
    "Agent": "High",
    "Aggregator": "Medium",
    "Partner": "Medium",
    "Branch": "Low",
    "Call Center": "Low"
}

channel["priority"] = channel["channel_name"].map(priority)

# ------------------------------------------------------------------
# Reorder Columns
# ------------------------------------------------------------------

channel = channel[
    [
        "channel_sk",
        "channel_id",
        "channel_name",
        "channel_type",
        "ownership",
        "priority",
        "is_digital"
    ]
]

# ------------------------------------------------------------------
# Save
# ------------------------------------------------------------------

channel.to_csv(
    GOLD_PATH / "dim_channel.csv",
    index=False
)

channel.to_parquet(
    GOLD_PATH / "dim_channel.parquet",
    index=False
)

print("✅ dim_channel created successfully")
print(channel)

✅ dim_channel created successfully
     channel_sk  channel_id channel_name channel_type ownership priority  \
0             1         1.0        Agent      Offline  Indirect     High   
1             2         3.0        Agent      Offline  Indirect     High   
2             3         4.0        Agent      Offline  Indirect     High   
3             4         6.0        Agent      Offline  Indirect     High   
4             5         7.0        Agent      Offline  Indirect     High   
..          ...         ...          ...          ...       ...      ...   
134         135       157.0    Corporate      Offline  Indirect      NaN   
135         136       158.0    Corporate      Offline  Indirect      NaN   
136         137       159.0    Corporate      Offline  Indirect      NaN   
137         138       160.0    Corporate      Offline  Indirect      NaN   
138         139       163.0    Corporate      Offline  Indirect      NaN   

     is_digital  
0         False  
1         False 

# dim_customer

In [5]:
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# Paths
# ---------------------------------------------------------

SILVER_PATH = Path("../data/silver")
GOLD_PATH = Path("../data/gold/dimensions")

GOLD_PATH.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# Load Customer
# ---------------------------------------------------------

customer = pd.read_csv(SILVER_PATH / "customer.csv")

# ---------------------------------------------------------
# Business Key
# ---------------------------------------------------------

customer["customer_id"] = [
    f"CUST{str(i).zfill(6)}"
    for i in range(1, len(customer)+1)
]

# ---------------------------------------------------------
# Driving License Status
# ---------------------------------------------------------

customer["driving_license_status"] = customer[
    "has_driving_license"
].map({
    1: "Licensed",
    0: "No License"
})

# ---------------------------------------------------------
# Customer Segment
# ---------------------------------------------------------

customer["customer_segment"] = customer[
    "previously_insured"
].map({
    1: "Existing Customer",
    0: "New Customer"
})

# ---------------------------------------------------------
# Insurance History
# ---------------------------------------------------------

customer["insurance_history"] = customer[
    "previously_insured"
].map({
    1: "Previously Insured",
    0: "First Time Buyer"
})

# ---------------------------------------------------------
# Risk Profile
# ---------------------------------------------------------

def risk(row):

    if row["has_driving_license"] == 0:
        return "High"

    if row["customer_age"] < 25:
        return "High"

    if row["customer_age"] > 60:
        return "Medium"

    return "Low"


customer["risk_profile"] = customer.apply(
    risk,
    axis=1
)

# ---------------------------------------------------------
# Final Columns
# ---------------------------------------------------------

customer = customer[
[
"customer_sk",
"customer_id",
"gender",
"customer_age",
"age_band",
"has_driving_license",
"driving_license_status",
"previously_insured",
"insurance_history",
"customer_segment",
"region",
"risk_profile"
]
]

# ---------------------------------------------------------
# Save
# ---------------------------------------------------------

customer.to_csv(
    GOLD_PATH/"dim_customer.csv",
    index=False
)

customer.to_parquet(
    GOLD_PATH/"dim_customer.parquet",
    index=False
)

print(customer.head())
print(f"\nRows : {len(customer):,}")

   customer_sk customer_id  gender  customer_age age_band  \
0            1  CUST000001    Male            20    18-25   
1            2  CUST000002    Male            20    18-25   
2            3  CUST000003  Female            20    18-25   
3            4  CUST000004  Female            20    18-25   
4            5  CUST000005  Female            20    18-25   

   has_driving_license driving_license_status  previously_insured  \
0                    1               Licensed                   0   
1                    1               Licensed                   0   
2                    1               Licensed                   1   
3                    1               Licensed                   1   
4                    1               Licensed                   0   

    insurance_history   customer_segment  region risk_profile  
0    First Time Buyer       New Customer    24.0         High  
1    First Time Buyer       New Customer    17.0         High  
2  Previously Insured  Exi

# fact_ai_interaction

In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
import uuid
import random

# ==========================================================
# Configuration
# ==========================================================

np.random.seed(42)
random.seed(42)

GOLD_PATH = Path("../data/gold")
FACT_PATH = GOLD_PATH / "facts"
DIM_PATH = GOLD_PATH / "dimensions"

FACT_PATH.mkdir(parents=True, exist_ok=True)

# ==========================================================
# Read Tables
# ==========================================================

quote = pd.read_csv(FACT_PATH / "fact_quote.csv")
customer = pd.read_csv(DIM_PATH / "dim_customer.csv")
date = pd.read_csv(DIM_PATH / "dim_date.csv")

# ----------------------------------------------------------
# Integrity check — replaces the previously unused customer read
# ----------------------------------------------------------
missing = set(quote["customer_sk"]) - set(customer["customer_sk"])
assert not missing, f"fact_quote references {len(missing)} customer_sk not in dim_customer"

# ==========================================================
# Date lookup — ties interaction date_sk/timestamp to the
# quote's actual date instead of a random pick from dim_date
# ==========================================================

date["date"] = pd.to_datetime(date["date"])
date_sk_to_date = dict(zip(date["date_sk"], date["date"]))
date_to_date_sk = dict(zip(date["date"], date["date_sk"]))
max_available_date = date["date"].max()

# ==========================================================
# Question Categories
# ==========================================================

question_bank = {
    "Premium": [
        "Why is my premium so high?",
        "Can I reduce my premium?",
        "How is premium calculated?",
        "Why did my premium increase?"
    ],
    "Documents": [
        "Which documents are required?",
        "How do I upload RC?",
        "Is Aadhaar mandatory?",
        "Can I upload later?"
    ],
    "Claims": [
        "How do I file a claim?",
        "Is bumper damage covered?",
        "How long does claim settlement take?",
        "What is cashless claim?"
    ],
    "Policy": [
        "Can I renew online?",
        "How do I download policy?",
        "How do I cancel policy?",
        "When does policy expire?"
    ],
    "Payment": [
        "Payment failed",
        "Can I pay later?",
        "Available payment methods?",
        "Can I use EMI?"
    ]
}

# ==========================================================
# AI Responses
# ==========================================================

responses = {
    "Premium": "Premium depends on vehicle, driver profile, location and claim history.",
    "Documents": "Please upload RC, Driving License, Aadhaar and previous policy copy.",
    "Claims": "You can register a claim online or through our 24x7 support.",
    "Policy": "Policy services are available through our website and mobile application.",
    "Payment": "We support UPI, Cards, Net Banking and Wallet payments."
}

# ==========================================================
# Generate AI Interaction Records
# ==========================================================

records = []
interaction_sk = 1

for _, row in quote.iterrows():

    quote_date = date_sk_to_date.get(row["date_sk"])
    interactions = np.random.randint(1, 4)

    for i in range(interactions):

        category = random.choice(list(question_bank.keys()))
        query = random.choice(question_bank[category])

        # ------------------------------------------------------
        # Timestamp/date_sk — derived from the quote's own date,
        # 0-2 days after (customer asking questions post-quote),
        # clipped to the last date the warehouse actually covers
        # ------------------------------------------------------
        interaction_date = quote_date + pd.Timedelta(days=np.random.randint(0, 3))
        interaction_date = min(interaction_date, max_available_date)
        interaction_timestamp = interaction_date + pd.Timedelta(
            hours=int(np.random.randint(8, 21)),
            minutes=int(np.random.randint(0, 60))
        )
        interaction_date_sk = date_to_date_sk[interaction_date]

        # ------------------------------------------------------
        # Correlated quality signals — confidence drives escalation,
        # escalation drives rating and sentiment, instead of three
        # independent random draws that contradict each other
        # ------------------------------------------------------
        confidence = round(np.random.uniform(0.55, 0.99), 2)
        escalation = bool(confidence < 0.75 or np.random.random() < 0.05)
        resolved = not escalation

        if escalation:
            rating = np.random.choice([1, 2, 3], p=[0.3, 0.4, 0.3])
            sentiment = np.random.choice(["Negative", "Neutral"], p=[0.7, 0.3])
        else:
            rating = np.random.choice([3, 4, 5], p=[0.1, 0.3, 0.6])
            sentiment = np.random.choice(["Positive", "Neutral", "Negative"], p=[0.7, 0.25, 0.05])

        response_time = np.random.randint(400, 2500)
        token_count = np.random.randint(120, 950)

        ai_version = np.random.choice(
            ["Claude Haiku 4.5", "Claude Sonnet 5", "Claude Opus 4.8"],
            p=[0.20, 0.55, 0.25]
        )

        records.append({
            "interaction_sk": interaction_sk,
            "customer_sk": row["customer_sk"],
            "quote_sk": row["quote_sk"],
            "date_sk": interaction_date_sk,
            "session_id": str(uuid.uuid4()),
            "interaction_timestamp": interaction_timestamp,
            "question_category": category,
            "user_query": query,
            "ai_response": responses[category],
            "response_time_ms": response_time,
            "confidence_score": confidence,
            "token_count": token_count,
            "escalation_required": escalation,
            "resolved_flag": resolved,
            "customer_rating": rating,
            "customer_sentiment": sentiment,
            "ai_model_version": ai_version
        })

        interaction_sk += 1

# ==========================================================
# Create DataFrame
# ==========================================================

fact_ai_interaction = pd.DataFrame(records)

# ==========================================================
# Response Speed Band
# ==========================================================

fact_ai_interaction["response_speed"] = pd.cut(
    fact_ai_interaction["response_time_ms"],
    bins=[0, 700, 1200, 2000, 5000],
    labels=["Excellent", "Good", "Average", "Slow"]
)

# ==========================================================
# AI Quality Score
# ==========================================================

fact_ai_interaction["ai_quality_score"] = (
    fact_ai_interaction["confidence_score"] * 100 * 0.7
    + fact_ai_interaction["customer_rating"] * 20 * 0.3
).round(2)

# ==========================================================
# Save
# ==========================================================

fact_ai_interaction.to_csv(FACT_PATH / "fact_ai_interaction.csv", index=False)
fact_ai_interaction.to_parquet(FACT_PATH / "fact_ai_interaction.parquet", index=False)

print("=" * 70)
print("fact_ai_interaction Created Successfully")
print("=" * 70)
print(fact_ai_interaction.head())
print(f"\nRows : {len(fact_ai_interaction):,}")
print(f"Columns : {len(fact_ai_interaction.columns)}")

fact_ai_interaction Created Successfully
   interaction_sk  customer_sk  quote_sk   date_sk  \
0               1            1         1  20250517   
1               2            1         1  20250518   
2               3            1         1  20250517   
3               4            2         2  20250926   
4               5            2         2  20250925   

                             session_id interaction_timestamp  \
0  aaebf48e-7735-4b69-8943-0e9885d8f8cc   2025-05-17 18:07:00   
1  f0f97977-046f-45b3-a371-2eb1cb438b53   2025-05-18 13:01:00   
2  92861815-f7d7-44af-b3ef-c7038db4d28c   2025-05-17 18:58:00   
3  4331fd11-e278-42df-86d1-e21b74705717   2025-09-26 19:54:00   
4  0717b1c4-46f0-447b-9103-ed5ea5da77f6   2025-09-25 09:57:00   

  question_category                  user_query  \
0           Premium  Why is my premium so high?   
1            Claims   Is bumper damage covered?   
2         Documents         How do I upload RC?   
3           Premium  Why is my premium 